# U02 - EDA
**Rossmann Store Sales**

**Mục tiêu:**
1. Khảo sát nhanh 3 file dữ liệu (train, test, store): shape, kiểu dữ liệu, anomalies.
2. Tìm các mối quan hệ/logic liên kết giữa các field.
3. Đặt giả thuyết (hypothesis) về **Business Problems**, phân biệt **root cause** vs **symptom**.
4. Kết nối lại thành 1 câu chuyện duy nhất → vì sao cần Mission 1 và Mission 2.
5. Ghi lại lưu ý/logic quan trọng cho Data Prep & Feature Engineering.


In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
pd.set_option('display.max_columns', 50)


## 0. Load dữ liệu & Data Dictionary tóm tắt

| File | Field | Kiểu | Ý nghĩa |
|---|---|---|---|
| train/test | Store | int | Mã cửa hàng (khóa nối với store.csv) |
| train/test | Date | date | Ngày |
| train/test | DayOfWeek | int (1-7) | Thứ trong tuần |
| train | Sales | int | Doanh số ngày đó (chỉ có ở train) |
| train | Customers | int | Số khách (chỉ có ở train) |
| train/test | Open | 0/1 | Cửa hàng có mở cửa ngày đó không |
| train/test | Promo | 0/1 | Khuyến mãi **ngắn hạn theo ngày** |
| train/test | StateHoliday | category | Ngày lễ nhà nước (0=không, a/b/c=loại lễ) |
| train/test | SchoolHoliday | 0/1 | Ngày lễ trường học |
| store | StoreType | a/b/c/d | Loại cửa hàng |
| store | Assortment | a/b/c | Mức độ đa dạng sản phẩm |
| store | CompetitionDistance | float | Khoảng cách tới đối thủ gần nhất (m) |
| store | CompetitionOpenSinceMonth/Year | float | Đối thủ mở cửa từ khi nào |
| store | Promo2 | 0/1 | Có tham gia khuyến mãi **định kỳ dài hạn** không (thuộc tính cố định của CH) |
| store | Promo2SinceWeek/Year | float | Tuần/năm bắt đầu tham gia Promo2 |
| store | PromoInterval | string | Các tháng mà Promo2 thực sự active trong năm |


In [ ]:
train = pd.read_csv('train_ross.csv', parse_dates=['Date'])
test  = pd.read_csv('test_ross.csv',  parse_dates=['Date'])
store = pd.read_csv('store_ross.csv')

print("train :", train.shape)
print("test  :", test.shape)
print("store :", store.shape)
print()
print("train date range:", train.Date.min().date(), "->", train.Date.max().date())
print("test  date range:", test.Date.min().date(),  "->", test.Date.max().date())
gap = (test.Date.min() - train.Date.max()).days
span_weeks = (test.Date.max() - test.Date.min()).days / 7
print(f"Khoảng cách train_max -> test_min: {gap} ngày | test span: {span_weeks:.1f} tuần")


**Nhận xét:** test.csv bắt đầu ngay sau train.csv (cách 1 ngày) và trải dài ~6.7 tuần. → Mission 2 phải dùng walk-forward split theo đúng cấu trúc thời gian này.

---
## 1. Data Anomalies
### 1.1. Missing values

In [ ]:
print("train.csv missing")
print(train.isna().sum()[train.isna().sum() > 0])
print("\ntest.csv missing")
print(test.isna().sum()[test.isna().sum() > 0])
print("\nstore.csv missing")
print(store.isna().sum())


In [ ]:
n_missing_interval = store['PromoInterval'].isna().sum()
n_promo2_0 = (store['Promo2']==0).sum()
print(f"PromoInterval missing: {n_missing_interval}  |  Promo2=0: {n_promo2_0}  |  Khớp: {n_missing_interval==n_promo2_0}")

**Nhận xét:**
- `train.csv` không có missing.
- `test.csv` thiếu 11 giá trị `Open` - **missing thật**, không thể loại bỏ (test là tập cần dự đoán)
- `store.csv` thiếu `CompetitionDistance` (3 dòng), `CompetitionOpenSinceMonth/Year` (354 dòng).
- `store.csv` thiếu `Promo2SinceWeek/Year`, `PromoInterval` đúng **544 dòng**, khớp chính xác số CH `Promo2=0` → **missing có cấu trúc**, không cần impute.


### 1.2. StateHoliday

In [ ]:
for v in train['StateHoliday'].unique():
    print(f"  value={v!r:8}  type={type(v).__name__}")
print()
print(train['StateHoliday'].value_counts(dropna=False))


In [ ]:
train['StateHoliday'] = train['StateHoliday'].astype(str).replace('0.0', '0')
test['StateHoliday']  = test['StateHoliday'].astype(str).replace('0.0', '0')
print(train['StateHoliday'].value_counts())


**Nhận xét:** '0' (string) và 0 (số) bị tách thành 2 nhóm khác nhau khi đọc bằng pandas → **bắt buộc ép về cùng 1 kiểu** trước khi dùng.

### 1.3. Logic Open ↔ Sales

In [ ]:
open0_sales_nonzero = ((train.Open==0) & (train.Sales!=0)).sum()
open1_sales_zero    = ((train.Open==1) & (train.Sales==0)).sum()
print(f"Open=0 nhưng Sales!=0: {open0_sales_nonzero} dòng (kỳ vọng 0)")
print(f"Open=1 nhưng Sales=0 : {open1_sales_zero} dòng (case lạ)")

anomaly = train[(train.Open==1) & (train.Sales==0)]
print("\nTheo DayOfWeek:"); print(anomaly['DayOfWeek'].value_counts().sort_index())


**Nhận xét:** 0 dòng vi phạm Open=0 → Sales=0 (Mục đích kiểm tra: Có ngày nào cửa hàng đóng cửa (Open = 0) mà vẫn phát sinh doanh số (Sales != 0) hay không?) → **bắt buộc loại Open=0 trước khi tính doanh số TB** (tránh mean bị kéo xuống). 54 dòng Open=1 & Sales=0 là outlier hiếm (~0.005%), rải rác nhiều ngày, không phải lễ (Mục đích kiểm tra: Có trường hợp nào ngược lại không: cửa hàng mở cửa (Open = 1) mà lại không bán được đồng nào (Sales = 0)?)

### 1.4. PromoInterval

In [ ]:
print(store['PromoInterval'].dropna().unique())
print()
print(store[store.Promo2==1]['PromoInterval'].value_counts())


**Nhận xét:** Toàn bộ 571 CH Promo2=1 chỉ rơi vào đúng 3 chu kỳ (`Jan,Apr,Jul,Oct` / `Feb,May,Aug,Nov` / `Mar,Jun,Sept,Dec`), phủ kín 12 tháng. Quan trọng cho phần 3.

### 1.5. Sample size theo StoreType / Assortment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
st_counts = store['StoreType'].value_counts().sort_index()
axes[0].bar(st_counts.index, st_counts.values, color='#4472C4')
for i,v in enumerate(st_counts.values): axes[0].text(i, v+10, str(v), ha='center')
axes[0].set_title('Số cửa hàng theo StoreType (tổng 1.115)')

as_counts = store['Assortment'].value_counts().sort_index()
axes[1].bar(as_counts.index, as_counts.values, color='#ED7D31')
for i,v in enumerate(as_counts.values): axes[1].text(i, v+10, str(v), ha='center')
axes[1].set_title('Số cửa hàng theo Assortment (tổng 1.115)')
plt.tight_layout(); plt.show()
print(st_counts); print(); print(as_counts)


**Nhận xét:** StoreType `b` (17 CH, 1.5%) và Assortment `b` (9 CH, 0.8%) quá nhỏ → mọi kết luận liên quan phải kèm n, nêu rõ "chỉ tham khảo".

### 1.6. Store coverage của test.csv 0 bao phủ hết 1.115 cửa hàng

In [ ]:
n_train_stores = train.Store.nunique()
n_test_stores  = test.Store.nunique()
missing_from_test = sorted(set(train.Store) - set(test.Store))
print(f"Số CH trong train: {n_train_stores}")
print(f"Số CH trong test : {n_test_stores}")
print(f"Số CH có trong train nhưng KHÔNG có trong test: {len(missing_from_test)}")
print(f"Số CH trong test nhưng không có trong train: {len(set(test.Store)-set(train.Store))}")
print(f"Vài mã CH bị thiếu trong test: {missing_from_test[:10]} ...")


**Phát hiện quan trọng, dễ bị bỏ sót:** `test.csv` chỉ có **856/1.115 CH**, thiếu **259 cửa hàng**.
→ **Ảnh hưởng trực tiếp đến Mission 2:** Model 2 chỉ cần (và chỉ có thể) tạo dự báo 6 tuần cho 856 CH có trong test, không phải toàn bộ 1.115 CH. Nếu vô tình build báo cáo "dự báo cho tất cả cửa hàng" thì sẽ sai phạm vi ngay từ đầu.

### 1.7. Đóng cửa dài ngày (khả năng do cải tạo/refurbishment)

In [ ]:
ts = train.sort_values(['Store','Date']).copy()
ts['closed'] = (ts.Open==0).astype(int)
ts['grp'] = (ts['closed'] != ts.groupby('Store')['closed'].shift()).cumsum()
runs = ts[ts.closed==1].groupby(['Store','grp']).size()

print("Phân phối độ dài các chuỗi ngày đóng cửa liên tục:")
print(runs.describe())
long_runs = runs[runs>=10]
n_stores_long = long_runs.reset_index()['Store'].nunique()
print(f"\nSố lần đóng cửa liên tục >=10 ngày: {(runs>=10).sum()}")
print(f"Số CH có ít nhất 1 lần đóng cửa dài (>=10 ngày): {n_stores_long}")
print(f"Chuỗi đóng cửa dài nhất: {runs.max()} ngày liên tục")


**Phát hiện quan trọng:** có **72 cửa hàng** từng đóng cửa liên tục ≥10 ngày, cá biệt có chuỗi lên tới **185 ngày** (~6 tháng), rất có thể là đóng cửa để cải tạo/sửa chữa.
→ **Ảnh hưởng đến các bước sau:**
- **Baseline forecast** (trung bình 4 tuần gần nhất): nếu 4 tuần gần nhất của 1 CH rơi đúng vào giai đoạn đóng cửa/mới mở lại, baseline sẽ méo mó nặng (quá ít ngày dữ liệu, hoặc sales tăng vọt bất thường ngay sau khi mở lại do nhu cầu dồn nén).
- **So sánh before-after Promo2**: nếu giai đoạn "active" hoặc "nghỉ" của 1 CH trùng với thời gian đóng cửa dài, tháng đó sẽ có rất ít ngày dữ liệu, dễ làm méo trung bình tháng.
→ Cần đánh dấu các CH/giai đoạn này để xử lý riêng (loại khỏi so sánh, hoặc dùng ngưỡng số ngày mở cửa tối thiểu/tháng).

### 1.8. CompetitionOpenSinceYear

In [ ]:
print(store['CompetitionOpenSinceYear'].describe())
print()
print("Các dòng có năm bất thường (<1990):")
print(store[store['CompetitionOpenSinceYear']<1990][['Store','CompetitionOpenSinceYear','CompetitionOpenSinceMonth']])


**Nhận xét:** phần lớn `CompetitionOpenSinceYear` nằm trong khoảng hợp lý (1990-2015), nhưng có vài giá trị như **1900, 1961** nhiều khả năng là giá trị placeholder/lỗi nhập liệu (đối thủ khó có thể "mở cửa từ năm 1900" trong bối cảnh dữ liệu bán lẻ hiện đại). → Nếu sau này dùng cột này để tạo feature "số năm đối thủ đã hoạt động", cần cap hoặc loại các giá trị < 1990 trước, tránh outlier làm lệch feature.

### 1.9. Duplicates


In [ ]:
# Kiểm tra khóa chính (Store, Date)
print(f"Duplicate (Store, Date) trong train: {train.duplicated(subset=['Store','Date']).sum()}")
print(f"Duplicate (Store, Date) trong test : {test.duplicated(subset=['Store','Date']).sum()}")
print("=> (Store, Date) là khóa chính duy nhất, mỗi dòng = đúng 1 CH x 1 ngày.")


**Xác nhận:** Không có dòng nào trùng lặp. Cặp `(Store, Date)` là khóa chính xác định duy nhất, 
mỗi dòng tương ứng với chính xác **1 cửa hàng x 1 ngày**. Dữ liệu sạch về mặt cấu trúc.


### 1.10. Tính liên tục thời gian

In [ ]:
# Kiểm tra xem mỗi CH có đủ 942 ngày (01/01/2013 -> 31/07/2015) không
days_per_store = train.groupby('Store')['Date'].count()
n_full = (days_per_store == 942).sum()
n_short = (days_per_store < 942).sum()

print(f"Giai đoạn train (01/01/2013 -> 31/07/2015) = 942 ngày.")
print(f"Số CH có đủ 942 ngày: {n_full}")
print(f"Số CH thiếu ngày    : {n_short} (min = {days_per_store.min()} ngày)")
print(f"\n10 CH có ít ngày nhất:")
print(days_per_store[days_per_store < 942].sort_values().head(10))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(days_per_store.values, bins=50, color='#4472C4', edgecolor='white')
ax.axvline(942, color='red', linestyle='--', linewidth=1.5, label=f'Full = 942 days ({n_full} CH)')
ax.axvline(days_per_store.min(), color='orange', linestyle='--', linewidth=1.5, label=f'Min = {days_per_store.min()} days ({n_short} CH thiếu)')
ax.set_xlabel('Số ngày có dữ liệu trong train')
ax.set_ylabel('Số cửa hàng')
ax.set_title(f'Time Continuity: {n_short}/1.115 CH thiếu ngày')
ax.legend()
plt.tight_layout(); plt.show()


**Phát hiện:** **181 CH chỉ có 758/942 ngày**: các CH này bắt đầu có dữ liệu muộn hơn (khoảng giữa năm 2013). -> Unbalanced dataset.

**=>** Khi tạo biến Lag / Rolling mean cho Model 2, cần xử lý khoảng trống này (nếu không sẽ bị NaN hoặc rò rỉ dữ liệu qua thời gian bất liên tục). 
Đồng thời khi tính baseline "trung bình 4 tuần gần nhất" cho các CH này ở giai đoạn đầu, cần có giá trị mặc định hợp lý.


### 1.11. Promo (ngắn hạn) chỉ chạy Thứ 2 đến Thứ 6


In [ ]:
# Kiểm tra Promo theo DayOfWeek
print(pd.crosstab(train['DayOfWeek'], train['Promo']))

==> `Promo` **chỉ chạy các ngày trong tuần (Thứ 2 đến Thứ 6)**, 
không có vào Thứ 7 hay Chủ Nhật. Đây là quy tắc cố định của Rossmann. 
Khi Feature Engineering hoặc khi predict trên test, **không được để model gán Promo=1 cho ngày cuối tuần**. 
Ngoài ra, phân tích tác động của Promo lên Sales phải chỉ so sánh trong các ngày T2-T6

### 1.12. missing Open thuộc duy nhất Store 622


In [ ]:
# Xác định chính xác 11 dòng missing Open trong test.csv
missing_open = test[test['Open'].isna()]
print(missing_open[['Id','Store','DayOfWeek','Date']])

s622 = train[train.Store == 622]
print(f"\nLịch sử Open của Store 622 theo DayOfWeek:")
print(s622.groupby('DayOfWeek')['Open'].mean())

**Phát hiện cụ thể:** Cả 11 dòng missing `Open` **đều thuộc duy nhất `Store 622`**, rơi vào các ngày từ Thứ 2 đến Thứ 7 (không có Chủ Nhật). 
Lịch sử hoạt động của Store 622 cho thấy: mở cửa **95-100% các ngày T2-T7**, đóng cửa **100% ngày CN**. 
→ **Rule chuẩn cho Data Prep: Impute `Open = 1` cho cả 11 dòng này.**


### 1.13. Đối thủ mới mở cửa trong giai đoạn nghiên cứu (2013-2015)


In [ ]:
# Kiểm tra bao nhiêu CH có đối thủ mới mở cửa TRONG giai đoạn 2013-2015
comp = store.dropna(subset=['CompetitionOpenSinceYear','CompetitionOpenSinceMonth']).copy()
comp['CompOpenDate'] = pd.to_datetime(
    comp['CompetitionOpenSinceYear'].astype(int).astype(str) + '-' + 
    comp['CompetitionOpenSinceMonth'].astype(int).astype(str) + '-01'
)
new_comp = comp[(comp['CompOpenDate'] >= '2013-01-01') & (comp['CompOpenDate'] <= '2015-09-30')]
print(f"Số CH có đối thủ MỚI mở cửa trong giai đoạn 2013-2015: {len(new_comp)} ({len(new_comp)/len(store)*100:.1f}%)")
print(f"\nPhân bố theo năm:")
print(new_comp['CompetitionOpenSinceYear'].value_counts().sort_index())


**Phát hiện:** **191 cửa hàng (17.1%)** có đối thủ cạnh tranh **mới mở cửa ngay trong giai đoạn dữ liệu train (2013-2015)**. 
==> Event Study: có thể đo lường chính xác doanh số sụt giảm bao nhiêu % ngay sau tháng đối thủ khai trương. 
Đồng thời, biến `CompetitionOpenSince` **biến động theo thời gian** cho 17% cửa hàng 
→ có thể derive feature `MonthsSinceCompetitionOpened` cho Model 2.


### 1.14. Phân phối Sales lệch phải -> ảnh hưởng RMSPE


In [ ]:
# Phân tích phân phối Sales và hiệu quả của log-transform
# Lưu ý: Lọc Open=1 & Sales>0 (loại 54 dòng ngoại lai Sales=0 đã phân tích ở mục 1.3)
open_sales_all = train[train.Open == 1]['Sales']
open_sales = train[(train.Open == 1) & (train.Sales > 0)]['Sales']

log_sales_all = np.log1p(open_sales_all)
log_sales = np.log1p(open_sales)

print(f"1. Sales gốc (Open=1 & Sales>0): mean={open_sales.mean():.0f}, median={open_sales.median():.0f}, skew={open_sales.skew():.2f}")
print(f"   - Max Sales: {train['Sales'].max():,}, Min Sales (Sales>0): {open_sales.min():,}")
print(f"\nPercentiles Sales (Open=1 & Sales>0):")
print(open_sales.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_string())

print(f"\n2. Hiệu quả sau log-transform (log(Sales + 1)):")
print(f"   - Nếu tính cả 54 dòng Sales=0: skew = {log_sales_all.skew():.2f} (bị 54 điểm log(1)=0 kéo lệch trái)")
print(f"   - Khi loại bỏ 54 dòng Sales=0 (chuẩn): skew giảm từ {open_sales.skew():.2f} xuống chỉ còn {log_sales.skew():.2f} (gần như đối xứng hoàn hảo!)")

# Trực quan hóa so sánh
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].hist(open_sales, bins=80, color='#4472C4', edgecolor='white', alpha=0.8)
axes[0].axvline(open_sales.mean(), color='red', linestyle='--', label=f'Mean = {open_sales.mean():.0f}')
axes[0].axvline(open_sales.median(), color='orange', linestyle='--', label=f'Median = {open_sales.median():.0f}')
axes[0].set_title(f'Phân phối Sales (Open=1 & Sales>0) | Skew = {open_sales.skew():.2f}')
axes[0].set_xlabel('Sales (€)'); axes[0].set_ylabel('Tần suất')
axes[0].legend()

axes[1].hist(log_sales, bins=80, color='#70AD47', edgecolor='white', alpha=0.8)
axes[1].axvline(log_sales.mean(), color='red', linestyle='--', label=f'Mean = {log_sales.mean():.2f}')
axes[1].axvline(log_sales.median(), color='orange', linestyle='--', label=f'Median = {log_sales.median():.2f}')
axes[1].set_title(f'Phân phối log(Sales+1) | Skew = {log_sales.skew():.2f}')
axes[1].set_xlabel('log(Sales + 1)'); axes[1].set_ylabel('Tần suất')
axes[1].legend()

plt.tight_layout()
plt.show()

**Phát hiện bản chất phân phối và giải thích chi tiết về Skewness:**
1. **Phân phối Sales gốc lệch phải mạnh (Skewness = +1.59):**
   - $\text{Mean} (6.956 €) > \text{Median} (6.369 €)$, với đuôi phải kéo rất dài lên tới giá trị tối đa $41.551 €$. 
2. **Hiệu quả của phép biến đổi Log (`log(Sales + 1)`):**
   - **Khi loại bỏ 54 dòng ngoại lai `Sales=0` (đã phân tích ở mục 1.3):** Hệ số Skewness giảm ngoạn mục từ **+1.59** xuống chỉ còn **-0.11** - phân phối trở nên hình chuông gần như đối xứng.
   - Nếu vẫn giữ 54 dòng `Open=1 & Sales=0`, giá trị $\log(0+1)=0$ nằm cách xa cụm phân phối chính (quanh 8.76) tới hơn 20 độ lệch chuẩn, tạo thành một đuôi nhọn kéo lệch về bên trái khiến skew bị âm thành $-0.64$. Điều này càng củng cố kết luận ở mục 1.3: *bắt buộc phải xử lý riêng 54 dòng dị biệt này*.

**Ảnh hưởng trực tiếp đến Model 2 (Forecasting):**
- Metric đánh giá của bài toán là **RMSPE** (Root Mean Squared Percentage Error), phạt sai số theo tỷ lệ phần trăm:
  $$\text{RMSPE} = \sqrt{\frac{1}{n}\sum \left(\frac{y - \hat{y}}{y}\right)^2}$$
- Dự báo lệch $500 €$ ở cửa hàng doanh số $2.000 €$ sẽ bị phạt nặng **gấp 5 lần** so với lệch $500 €$ ở cửa hàng doanh số $10.000 €$.
- Do $\log(y) - \log(\hat{y}) = \log\left(\frac{y}{\hat{y}}\right) \approx \frac{y - \hat{y}}{\hat{y}}$, việc tối ưu hàm mất mát trên không gian log (`log(Sales+1)`) tương thích tự nhiên với bản chất phần trăm của RMSPE và đồng thời khắc phục triệt để hiện tượng phân phối lệch phải.
- **Quy tắc hậu xử lý bắt buộc:** Huấn luyện model trên `log(Sales+1)` cho các ngày mở cửa, sau đó lấy $\exp(\hat{y}) - 1$, và **bắt buộc gán đè $\text{Sales} = 0$ khi $\text{Open} = 0$**.

### 1.15. 225 CH có dữ liệu trước VÀ sau khi tham gia Promo2 trong train


In [ ]:
# Xác định bao nhiêu CH tham gia Promo2 TRONG giai đoạn train (có dữ liệu before & after)
p2 = store[store.Promo2 == 1].copy()
p2['Promo2StartDate'] = pd.to_datetime(
    p2['Promo2SinceYear'].astype(int).astype(str) + '-W' + 
    p2['Promo2SinceWeek'].astype(int).astype(str).str.zfill(2) + '-1', 
    format='%Y-W%W-%w'
)
before = (p2['Promo2StartDate'] < '2013-01-01').sum()
during = ((p2['Promo2StartDate'] >= '2013-01-01') & (p2['Promo2StartDate'] <= '2015-07-31')).sum()

print(f"- Tham gia TRƯỚC 01/01/2013 (chỉ có 'after' trong train): {before} CH")
print(f"- Tham gia TRONG giai đoạn train 2013-2015 (có cả before & after): {during} CH")
print(f"\n=> {during} CH có dữ liệu trước VÀ sau khi gia nhập Promo2 ngay trong train.csv")
print(f"-> Nhóm quý giá nhất để phân tích nhân quả (before-after / DiD) cho Mission 1.")


**Phát hiện quan trọng cho Mission 1:** Trong 571 CH tham gia Promo2:
- **346 CH** đã tham gia từ trước khi train bắt đầu (01/01/2013) chỉ có dữ liệu giai đoạn 'sau khi tham gia', không có baseline 'trước khi tham gia'.
- **225 CH** tham gia **ngay trong giai đoạn train** có cả dữ liệu **trước và sau** khi gia nhập Promo2.

Nhóm 225 CH này là **nhóm quý giá nhất** để phân tích nhân quả: có thể dùng Difference-in-Differences (DiD) 
hoặc so sánh within-store trước/sau mốc tham gia, kiểm soát được cả yếu tố cố định của từng CH. 
Mục 3 (Điều tra Promo2) cần tập trung vào nhóm này để có kết luận chắc chắn nhất.


---
## 2. Mối quan hệ giữa các field (Relate)

Từ đây, mọi phân tích doanh số đều **loại Open=0** trước, và **merge store vào train**.


In [ ]:
df = train[train.Open==1].merge(store, on='Store', how='left')
print("Shape sau khi lọc Open=1 và merge store:", df.shape)


### 2.1. Promo (khuyến mãi ngắn hạn) vs Sales

In [ ]:
promo_cmp = df.groupby('Promo')['Sales'].agg(['mean','count'])
pct = (promo_cmp.loc[1,'mean'] / promo_cmp.loc[0,'mean'] - 1) * 100

fig, ax = plt.subplots(figsize=(5,4))
bars = ax.bar(['Không Promo','Có Promo'], promo_cmp['mean'], color=['#A6A6A6','#4472C4'])
for b, n in zip(bars, promo_cmp['count']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+80, f"n={n:,}", ha='center')
ax.set_ylabel('Sales trung bình / ngày')
ax.set_title(f'Promo ngắn hạn: +{pct:.1f}% doanh số')
plt.tight_layout(); plt.show()
print(promo_cmp); print(f"Chênh lệch: {pct:.1f}%")


**Nhận xét:** Ngày có Promo ngắn hạn, Sales TB cao hơn rõ rệt (~+38%), mẫu rất lớn ở cả 2 nhóm → tín hiệu tương quan cực mạnh, là feature then chốt cho Model 2 (dự báo).
- **Lưu ý về bản chất Nhân quả (Causation) vs Tương quan (Correlation):** Mức tăng +38% này là tín hiệu dự báo rất tốt cho ngày mai, nhưng **không đồng nghĩa với việc Promo tạo ra 38% tăng trưởng ròng cho toàn chu kỳ**. Trong bán lẻ, một phần lớn mức tăng này thường đến từ **hiệu ứng dịch chuyển thời gian mua sắm (Forward-buying / Cannibalization)**: khách hàng quen biết trước lịch giảm giá nên nhịn mua tuần trước và mua tích trữ vào tuần khuyến mãi. Muốn đánh giá hiệu quả sinh lời thực tế của chính sách giá (Mission 1), cần xem xét tổng doanh số và biên lợi nhuận theo chu kỳ tuần/tháng, tránh kết luận nhân quả ngây thơ.

### 2.2. CompetitionDistance vs Sales

In [ ]:
d = df.dropna(subset=['CompetitionDistance']).copy()
corr = d['CompetitionDistance'].corr(d['Sales'])
print(f"Tương quan CompetitionDistance vs Sales: {corr:.4f}")

bins = [0, 500, 1000, 2500, 5000, 10000, 1e9]
labels = ['<500m','500-1000m','1-2.5km','2.5-5km','5-10km','>10km']
d['dist_bin'] = pd.cut(d['CompetitionDistance'], bins=bins, labels=labels)
g = d.groupby('dist_bin')['Sales'].agg(['mean','count'])

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(g.index.astype(str), g['mean'], color='#70AD47')
for i,(m,n) in enumerate(zip(g['mean'], g['count'])):
    ax.text(i, m+80, f"n={n:,}", ha='center', fontsize=8)
ax.set_ylabel('Sales trung bình / ngày'); ax.set_title('Sales TB theo khoảng cách đối thủ')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()
print(g)


**Giả thuyết bị bác bỏ:** tương quan chỉ ≈ -0.036, không có xu hướng đơn điệu rõ ràng theo nhóm khoảng cách → `CompetitionDistance` không phải yếu tố giải thích chính cho biến động doanh số.

### 2.3. StoreType & Assortment vs Sales

In [ ]:
g_st = df.groupby('StoreType')['Sales'].mean(); n_st = store.groupby('StoreType').size()
g_as = df.groupby('Assortment')['Sales'].mean(); n_as = store.groupby('Assortment').size()

fig, axes = plt.subplots(1,2, figsize=(11,4))
bars = axes[0].bar(g_st.index, g_st.values, color='#4472C4')
for b, n in zip(bars, n_st[g_st.index]): axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+80, f"n={n}", ha='center')
axes[0].set_title('Sales TB theo StoreType')

bars2 = axes[1].bar(g_as.index, g_as.values, color='#ED7D31')
for b, n in zip(bars2, n_as[g_as.index]): axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+80, f"n={n}", ha='center')
axes[1].set_title('Sales TB theo Assortment')
plt.tight_layout(); plt.show()


**Nhận xét:** StoreType `b` (n=17) và Assortment `b` (n=9) cao vượt trội, rất có thể là hiệu ứng nhóm nhỏ, không đại diện tổng quát. Nhóm `a`/`c`/`d` (StoreType) hay `a`/`c` (Assortment) đáng tin hơn nhiều (hàng trăm CH).

In [ ]:
# Đào sâu bản chất StoreType b: Khách hàng, Giỏ hàng (Basket size) và Lịch mở cửa Chủ nhật
df['BasketSize'] = df['Sales'] / df['Customers']

# Tỷ lệ mở cửa ngày Chủ nhật (tính trên tập train gốc gồm cả Open=0)
train_st = train[train.DayOfWeek == 7].merge(store[['Store', 'StoreType']], on='Store')
sun_open_rate = train_st.groupby('StoreType')['Open'].mean() * 100

st_summary = pd.DataFrame({
    'Số CH (n)': store.groupby('StoreType').size(),
    'Tỷ trọng CH (%)': (store.groupby('StoreType').size() / len(store) * 100).round(1),
    'Tỷ lệ mở CN (%)': sun_open_rate.round(1),
    'Khách TB / ngày': df.groupby('StoreType')['Customers'].mean().round(0).astype(int),
    'Giỏ hàng TB (€)': df.groupby('StoreType')['BasketSize'].mean().round(2),
    'Sales TB / ngày (€)': df.groupby('StoreType')['Sales'].mean().round(0).astype(int)
})

print("=== BẢNG ĐẶC ĐIỂM STORETYPE ===")
print(st_summary)
print("\n* Lưu ý: Toàn bộ 9 cửa hàng Assortment 'b' (extra) đều thuộc StoreType 'b'.")

# Trực quan hóa so sánh 3 khía cạnh: Lượng khách, Giỏ hàng và Tỷ lệ mở Chủ nhật
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

bars1 = axes[0].bar(st_summary.index, st_summary['Khách TB / ngày'], color=['#4472C4', '#ED7D31', '#4472C4', '#4472C4'])
for b in bars1: axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+35, f"{int(b.get_height()):,}", ha='center', fontsize=9)
axes[0].set_title('Khách TB / ngày (khi Open=1)')
axes[0].set_ylabel('Số khách')

bars2 = axes[1].bar(st_summary.index, st_summary['Giỏ hàng TB (€)'], color=['#4472C4', '#ED7D31', '#4472C4', '#4472C4'])
for b in bars2: axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.2, f"{b.get_height():.2f}€", ha='center', fontsize=9)
axes[1].set_title('Giá trị giỏ hàng TB (Sales/Customers)')
axes[1].set_ylabel('EUR (€)')

bars3 = axes[2].bar(st_summary.index, st_summary['Tỷ lệ mở CN (%)'], color=['#4472C4', '#ED7D31', '#4472C4', '#4472C4'])
for b in bars3: axes[2].text(b.get_x()+b.get_width()/2, b.get_height()+2, f"{b.get_height():.1f}%", ha='center', fontsize=9)
axes[2].set_title('Tỷ lệ mở cửa ngày Chủ nhật (%)')
axes[2].set_ylabel('%')

plt.tight_layout()
plt.show()


**Đào sâu bản chất: Liệu StoreType `b` / Assortment `b` có phải là phân khúc 'Premium' (cao cấp)?**

1. **Bản chất thực tế: Cửa hàng nhà ga / sân bay (Transit Stores), KHÔNG phải Premium:**
   - Nếu là cửa hàng cao cấp (premium), giá trị mỗi đơn hàng (Basket size) phải lớn. Nhưng thực tế nhóm `b` có giỏ hàng **thấp nhất toàn hệ thống (chỉ 5.13 €/khách)**, trong khi nhóm `d` mới là phân khúc mua sắm giỏ hàng lớn (11.28 €/khách).
   - Doanh số tổng của nhóm `b` cao vượt trội hoàn toàn đến từ: **Lượng khách cực khủng (~2,022 khách/ngày**, gấp 2.5–3 lần các nhóm khác) và **mở cửa 98.4% vào các ngày Chủ nhật** (luật Đức cấm hầu hết cửa hàng mở CN, ngoại trừ các cửa hàng tiện ích ở đầu mối giao thông như Hauptbahnhof/Sân bay).
   - Toàn bộ 9 cửa hàng `Assortment b` (extra) đều nằm trọn trong 17 cửa hàng `StoreType b` này.

2. **Vì sao trong thống kê lại kết luận là 'không đáng tin cậy' (Lack of Statistical Reliability)?**
   - **Cỡ mẫu quá nhỏ (Sample Size Bias):** Toàn hệ thống có 1.115 cửa hàng nhưng StoreType `b` chỉ có **17 CH (1.5%)** và Assortment `b` chỉ có **9 CH (0.8%)**. Độ biến động (variance) của nhóm nhỏ rất lớn, chỉ 1 vài cửa hàng vị trí đắc địa tăng đột biến sẽ làm lệch hoàn toàn trung bình nhóm.
   - **Nguy cơ sai lệch khi suy rộng (Generalization Fallacy):** Vì doanh số cao là do đặc thù vị trí địa bàn (nhà ga, mở xuyên ngày nghỉ), ta **không thể khái quát hóa** thành quy luật kinh doanh: *'Cứ đổi sang StoreType b hay mở rộng Assortment b là sẽ đạt doanh số 10.000 €/ngày'*. Bạn không thể mở hàng trăm cửa hàng ở nhà ga được.
   - Do đó, khi đánh giá nguyên nhân doanh số hay đưa vào mô hình hóa (Mission 1 & Mission 2), số liệu nhóm `b` bắt buộc phải kèm cỡ mẫu $n$ và chỉ xem như trường hợp cá biệt, không lấy làm đại diện chung.

### 2.4. DayOfWeek

In [ ]:
open_rate = train.groupby('DayOfWeek')['Open'].mean()
sales_by_dow = df.groupby('DayOfWeek')['Sales'].mean()

fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].bar(open_rate.index, open_rate.values, color='#4472C4')
axes[0].set_title('Tỷ lệ mở cửa theo DayOfWeek (1=Mon...7=Sun)')
axes[0].set_ylabel('Tỷ lệ Open')

axes[1].bar(sales_by_dow.index, sales_by_dow.values, color='#ED7D31')
axes[1].set_title('Sales TB theo DayOfWeek (khi Open=1)')
plt.tight_layout(); plt.show()
print("Tỷ lệ Open:\n", open_rate.round(3))
print("\nSales TB (Open=1):\n", sales_by_dow)


**Phát hiện:** Chủ nhật (DayOfWeek=7) chỉ mở cửa **2.5%** số lần, phần lớn cửa hàng đóng cửa Chủ nhật (đúng quy định bán lẻ tại Đức). Nhưng những lần **hiếm hoi** mở cửa Chủ nhật lại có Sales TB rất cao (đứng thứ 2 sau Thứ 2). Thứ 7 tuy là cuối tuần nhưng lại là ngày có Sales TB **thấp nhất**. → `DayOfWeek` là driver rất mạnh cho forecast, nhưng ngày Chủ nhật hiếm hoi có thể là các trường hợp đặc biệt (mở bán dịp lễ), không nên coi Sales trung bình ngày Chủ nhật là "tiêu biểu" vì mẫu quá nhỏ (chỉ 2.5% số ngày).

### 2.5. SchoolHoliday vs Sales

In [ ]:
g = df.groupby('SchoolHoliday')['Sales'].agg(['mean','count'])
print(g)
pct = (g.loc[1,'mean']/g.loc[0,'mean']-1)*100
print(f"Chênh lệch: {pct:+.1f}%")


**Nhận xét:** Sales TB ngày có SchoolHoliday cao hơn nhẹ (~+4.4%), mẫu đủ lớn ở cả 2 nhóm. Tín hiệu nhất quán → thích hợp làm feature bổ trợ cho Model 2.
- **Cảnh báo ngộ nhận tương quan:** Tương tự như `StateHoliday`, mức tăng +4.4% này chỉ là **con số bình quân gộp**. Kỳ nghỉ học thực tế có thể thúc đẩy sức mua ở các cửa hàng gần khu dân cư / điểm vui chơi, nhưng lại làm sụt giảm ở các cửa hàng gần trường đại học, trường học hoặc khu văn phòng công sở. Không nên ngộ nhận là cứ nghỉ học thì toàn bộ 1.115 cửa hàng đều tăng doanh số như nhau.

### 2.6. StateHoliday vs Sales, chỉ khi cửa hàng thực sự mở cửa (rất hiếm)

In [ ]:
print("Tỷ lệ Open theo StateHoliday:")
print(train.groupby('StateHoliday')['Open'].mean())
print()
g = df.groupby('StateHoliday')['Sales'].agg(['mean','count'])
print("Sales TB khi Open=1, theo loại StateHoliday:")
print(g)


**Nhận xét:** Cửa hàng gần như luôn đóng cửa vào ngày lễ (tỷ lệ Open chỉ 1.7-3.4% cho loại a/b/c). Nhưng trong số ít trường hợp mở cửa đúng ngày lễ, Sales TB **cao vượt trội** (8.487-9.888 so với 6.954 ngày thường). Có thể là các CH ở vị trí đặc thù (ga tàu, khu du lịch) được phép mở cửa và hưởng nhu cầu dồn nén. Mẫu rất nhỏ (71-694 dòng) nên chỉ nêu như quan sát thú vị, **không nên dùng để suy rộng** ra toàn bộ hệ thống.

In [ ]:
# Đào sâu bản chất ngày lễ: Bóc tách doanh số theo từng StoreType để kiểm tra giả thuyết
df['IsHoliday'] = df['StateHoliday'] != '0'

st_comp = df.groupby(['StoreType', 'IsHoliday'])['Sales'].agg(['count', 'mean']).unstack()
st_comp.columns = ['Count_Normal', 'Count_Holiday', 'Sales_Normal', 'Sales_Holiday']
st_comp['Holiday_Pct_of_Total_Open'] = (st_comp['Count_Holiday'] / st_comp['Count_Holiday'].sum() * 100).round(1)
st_comp['Sales_Diff_%'] = ((st_comp['Sales_Holiday'] - st_comp['Sales_Normal']) / st_comp['Sales_Normal'] * 100).round(1)

summary_display = pd.DataFrame({
    'Số lượt mở ngày lễ': st_comp['Count_Holiday'],
    'Tỷ trọng trong ngày lễ (%)': st_comp['Holiday_Pct_of_Total_Open'],
    'Sales TB Ngày thường (€)': st_comp['Sales_Normal'].round(0).astype(int),
    'Sales TB Ngày lễ (€)': st_comp['Sales_Holiday'].round(0).astype(int),
    'Chênh lệch (%)': st_comp['Sales_Diff_%']
})

print("=== BẢNG BÓC TÁCH DOANH SỐ NGÀY LỄ VS NGÀY THƯỜNG THEO STORETYPE ===")
print(summary_display)

# Trực quan hóa Nghịch lý Simpson trong ngày lễ
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(len(st_comp.index))
width = 0.35

bars1 = axes[0].bar(x - width/2, st_comp['Sales_Normal'], width, label='Ngày thường (StateHoliday=0)', color='#4472C4')
bars2 = axes[0].bar(x + width/2, st_comp['Sales_Holiday'], width, label='Ngày lễ (StateHoliday a/b/c)', color='#ED7D31')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"StoreType {st}" for st in st_comp.index])
axes[0].set_ylabel('Sales TB / ngày (€)')
axes[0].set_title("Nghịch lý Simpson: Chỉ StoreType 'b' tăng, 'a' & 'd' giảm mạnh ngày lễ")
axes[0].legend()
for b in bars1: axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+120, f"{int(b.get_height()):,}", ha='center', fontsize=8.5)
for b in bars2: axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+120, f"{int(b.get_height()):,}", ha='center', fontsize=8.5, fontweight='bold')

bars3 = axes[1].bar(st_comp.index, st_comp['Holiday_Pct_of_Total_Open'], color=['#A6A6A6', '#C00000', '#A6A6A6', '#A6A6A6'])
axes[1].set_title("Cơ cấu mở cửa ngày lễ: StoreType 'b' chiếm gần 50% số lượt")
axes[1].set_ylabel('% trong tổng số lượt mở ngày lễ')
axes[1].set_xlabel('StoreType')
for b in bars3: axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+1, f"{b.get_height():.1f}%", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


**Đào sâu chứng minh bản chất: Bóc trần Nghịch lý Simpson (Simpson's Paradox) trong ngày lễ:**

1. **Cửa hàng thông thường (StoreType `a` & `d`) bị SỤT GIẢM doanh số rất mạnh khi mở ngày lễ:**
   - StoreType `a` (khu dân cư): Doanh số ngày lễ **giảm -27.4%** (từ 6.926 € xuống còn 5.027 €). Lượng khách giảm từ 795 xuống 613 người.
   - StoreType `d`: Doanh số ngày lễ **giảm -9.9%** (từ 6.822 € xuống còn 6.145 €). Đặc biệt vào dịp Lễ Phục sinh (Easter) giảm xuống 3.092 € và Giáng sinh (Christmas) chỉ còn vỏn vẹn 1.800 €.
   - **Thực tế nghiệp vụ:** Vào ngày lễ nhà nước, người dân nghỉ ngơi, đi du lịch hoặc về quê, các khu dân cư và trung tâm thương mại thông thường vắng bóng người, khiến các cửa hàng loại `a` và `d` nếu có mở cửa cũng ế ẩm hơn hẳn ngày thường.

2. **StoreType `b` (Cửa hàng nhà ga/sân bay) là động cơ duy nhất kéo trung bình lên:**
   - Ngược lại hoàn toàn với `a` và `d`, StoreType `b` vào ngày lễ lại **bùng nổ doanh số (+17.9%)**, tăng từ 10.179 € lên **12.002 €/ngày** (Lễ Phục sinh thậm chí đạt đỉnh **13.038 €**). Lý do là dòng người đổ xô về các ga tàu, sân bay để di chuyển, đi chơi lễ.
   - **Điểm mấu chốt quyết định con số tổng:** Mặc dù StoreType `b` chỉ chiếm vỏn vẹn **1.5%** tổng số cửa hàng toàn chuỗi, nhưng trong số các ngày lễ có mở cửa, **StoreType `b` chiếm tới 49.6% (gần một nửa) tổng số lượt mở** (451 / 910 lượt)!

3. **Kết luận khoa học dữ liệu:**
   - Con số *'Sales TB ngày lễ cao hơn ngày thường (8.809 € vs 6.954 €)'* ở bảng tổng gộp bên trên là một **ảo giác thống kê do lệch cơ cấu mẫu (Composition Effect / Simpson's Paradox)**.
   - Bản thân 'ngày lễ' không làm tăng doanh số cho hệ thống bán lẻ Rossmann, mà chỉ làm tăng doanh số cho nhóm cửa hàng đầu mối giao thông (nhà ga xe lửa/sân bay). Với hơn 95% cửa hàng còn lại, mở cửa ngày lễ thậm chí gây lỗ vì doanh số sụt giảm trong khi chi phí nhân công ngày lễ cao gấp bội.

### 2.7. Customers vs Sales

In [ ]:
corr_cs = df['Sales'].corr(df['Customers'])
print(f"Tương quan Sales vs Customers: {corr_cs:.3f}")

d = df.copy()
d['basket'] = d['Sales'] / d['Customers'].replace(0, np.nan)
print("\nBasket size (Sales/Customers) — thống kê mô tả:")
print(d['basket'].describe())

fig, ax = plt.subplots(figsize=(6,4))
ax.hist(d['basket'].dropna(), bins=60, range=(0,25), color='#4472C4', edgecolor='white')
ax.set_xlabel('Basket size (Sales / Customers), đơn vị tiền/khách')
ax.set_ylabel('Số dòng (ngày-cửa hàng)')
ax.set_title('Phân phối basket size toàn hệ thống')
plt.tight_layout(); plt.show()


**Nhận xét:** Sales và Customers tương quan rất mạnh (r≈0.82). Dễ hiểu vì khách càng đông thì doanh số càng cao. Nhưng **basket size** (Sales/Customers, tức mỗi khách chi trung bình bao nhiêu) lại là **1 chỉ số độc lập với lưu lượng khách** vì 1 CH có thể ít khách nhưng khách chi nhiều (basket cao) hoặc đông khách nhưng chi ít (basket thấp). → Bổ sung cho câu hỏi Mission 1: nếu 1 CH có basket size thấp lâu dài (không đổi trước/sau Promo2), có thể gợi ý vấn đề nằm ở **định vị/product-market fit của cửa hàng đó** (khách vào nhưng không chi nhiều), khác hẳn với vấn đề "khuyến mãi không hiệu quả". Lưu ý: `Customers` **không có trong test.csv** nên chỉ dùng được cho phân tích mô tả (Mission 1), không dùng được làm feature cho Model 2 (forecast).

---
## 3. Điều tra Promo2: Symptom, Hypothesis & Root Cause

**Symptom (điều Giám đốc Vận hành quan sát thấy):** 571 CH tham gia Promo2 có doanh số TB thấp hơn 544 CH không tham gia.
**Câu hỏi:** đây có phải bằng chứng "Promo2 làm giảm doanh số", hay chỉ là hiện tượng bề mặt che giấu nguyên nhân khác?


### 3.1. Tái hiện con số của Giám đốc Vận hành (so sánh cross-sectional thô)

In [ ]:
store_level = df.groupby(['Store','Promo2'])['Sales'].mean().reset_index()
naive = store_level.groupby('Promo2')['Sales'].agg(['mean','count'])
naive.index = ['Promo2 = 0','Promo2 = 1']
naive.columns = ['Sales TB / ngày (theo CH)','Số CH (n)']
gap_pct = (naive.iloc[1,0]/naive.iloc[0,0]-1)*100

fig, ax = plt.subplots(figsize=(5,4))
bars = ax.bar(naive.index, naive['Sales TB / ngày (theo CH)'], color=['#A6A6A6','#C00000'])
for b, n in zip(bars, naive['Số CH (n)']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+80, f"n={n}", ha='center')
ax.set_ylabel('Sales TB/ngày (trung bình theo CH)')
ax.set_title(f'So sánh thô: Promo2=1 thấp hơn {abs(gap_pct):.1f}%')
plt.tight_layout(); plt.show()
print(naive); print(f"\nChênh lệch: {gap_pct:.1f}%")


**Đã tái hiện đúng con số của HĐQT:** chênh lệch khoảng **10-11%**. Đây là **symptom**. Câu hỏi thật là **root cause**: tại sao có chênh lệch này?

### 3.2. Hypothesis 1 - Selection bias: 2 nhóm CH vốn dĩ khác nhau về đặc điểm nền?

In [ ]:
ct_st = pd.crosstab(store.Promo2, store.StoreType, normalize='index')*100
ct_as = pd.crosstab(store.Promo2, store.Assortment, normalize='index')*100
comp_dist = store.groupby('Promo2')['CompetitionDistance'].mean()

fig, axes = plt.subplots(1,3, figsize=(14,4))
ct_st.T.plot(kind='bar', ax=axes[0], color=['#A6A6A6','#C00000'])
axes[0].set_title('% StoreType theo nhóm Promo2'); axes[0].legend(['Promo2=0','Promo2=1'])
ct_as.T.plot(kind='bar', ax=axes[1], color=['#A6A6A6','#C00000'])
axes[1].set_title('% Assortment theo nhóm Promo2'); axes[1].legend(['Promo2=0','Promo2=1'])
axes[2].bar(['Promo2=0','Promo2=1'], comp_dist.values, color=['#A6A6A6','#C00000'])
axes[2].set_title('CompetitionDistance TB theo nhóm Promo2')
plt.tight_layout(); plt.show()

print("StoreType %:\n", ct_st.round(1))
print("\nAssortment %:\n", ct_as.round(1))
print("\nCompetitionDistance TB:\n", comp_dist)


**Nhận xét:**
- **Assortment**: phân bố gần như giống hệt nhau giữa 2 nhóm → không giải thích được chênh lệch.
- **StoreType**: nhóm Promo2=1 có tỷ lệ StoreType `b` (bán chạy nhất, mục 2.3) thấp hơn hẳn (0.9% vs 2.2%) → giải thích được **một phần nhỏ**.
- **CompetitionDistance**: nhóm Promo2=1 gần đối thủ hơn (4.317m vs 6.550m) — nhưng mục 2.2 đã cho thấy biến này gần như không tương quan với Sales → khác biệt có thật nhưng không phải nguyên nhân.

→ Composition 2 nhóm khác nhau đôi chút nhưng **không đủ giải thích hết** 10-11% chênh lệch.
- **Lưu ý phương pháp luận (Observable vs Unobserved Confounders):** Kết quả này chỉ cho thấy Selection Bias **không rõ rệt trên các biến nền quan sát được trong `store.csv`**. Tuy nhiên, trong thực tế kinh doanh vẫn có thể tồn tại **các yếu tố gây nhiễu không quan sát được (Unobserved Confounders)** — ví dụ: năng lực quản lý của cửa hàng trưởng, vị trí mặt bằng vi mô, hoặc xu hướng kinh doanh sa sút từ trước khiến cửa hàng quyết định tham gia Promo2 để cứu vãn. Đó là lý do tại Mission 1, phương pháp phân tích nhân quả tối ưu nhất là dùng mô hình **Difference-in-Differences (DiD)** trên **225 cửa hàng mới gia nhập Promo2** (mục 1.15) để kiểm soát triệt để các đặc tính cố định của từng cửa hàng.

### 3.3. Xây dựng biến `IsPromo2ActiveThisMonth` (bước chuẩn bị quan trọng nhất)

In [ ]:
month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,
             'Sep':9,'Sept':9,'Oct':10,'Nov':11,'Dec':12}

def parse_interval(s):
    if pd.isna(s): return set()
    return set(month_map[m.strip()] for m in s.split(','))

d2 = df[df.Promo2==1].copy()
d2['interval_months'] = d2['PromoInterval'].map(parse_interval)
d2['Year']  = d2.Date.dt.isocalendar().year
d2['Week']  = d2.Date.dt.isocalendar().week
d2['Month'] = d2.Date.dt.month

def past_since(row):
    y, w = row['Year'], row['Week']
    sy, sw = row['Promo2SinceYear'], row['Promo2SinceWeek']
    if pd.isna(sy) or pd.isna(sw): return False
    return (y > sy) or (y == sy and w >= sw)

d2['past_since']         = d2.apply(past_since, axis=1)
d2['month_in_interval']  = d2.apply(lambda r: r['Month'] in r['interval_months'], axis=1)
d2['IsPromo2ActiveThisMonth'] = (d2['past_since'] & d2['month_in_interval']).astype(int)

print("Điều kiện: Promo2=1 AND đã qua mốc SinceWeek/Year AND tháng nằm trong PromoInterval")
print(d2['IsPromo2ActiveThisMonth'].value_counts())


**Test case thủ công**

In [ ]:
row = store[store.Store==6].iloc[0]
print(f"Store 6: Promo2={row.Promo2}, SinceWeek={row.Promo2SinceWeek}, "
      f"SinceYear={row.Promo2SinceYear}, PromoInterval={row.PromoInterval}")
print()
sample = d2[d2.Store==6][['Date','Year','Week','Month','past_since','month_in_interval','IsPromo2ActiveThisMonth']]
print(sample[sample.Month.isin([1,2,3,4])].drop_duplicates('Month').sort_values('Date'))


**Nhận xét:** logic hoạt động đúng. `month_in_interval` chỉ True ở đúng các tháng trong PromoInterval của CH đó; `IsPromo2ActiveThisMonth` chỉ =1 khi cả 2 điều kiện cùng đúng. Đây là biến bắt buộc. Nếu chỉ dùng "trước/sau khi tham gia" thô, giai đoạn "sau" sẽ lẫn cả tháng active lẫn nghỉ, làm loãng tín hiệu.

### 3.4. So sánh within-store: tháng active vs tháng nghỉ (chưa xét mùa vụ)

In [ ]:
post = d2[d2['past_since']].copy()

# cảnh báo: loại các store-month có quá ít ngày mở cửa (khả năng do đóng cửa dài — mục 1.7)
days_per_sm = post.groupby(['Store','Year','Month']).size()
thin = days_per_sm[days_per_sm<5]
print(f"Số tháng-CH có <5 ngày mở cửa (khả năng do đóng cửa dài hạn): {len(thin)}/{len(days_per_sm)}")
post = post.set_index(['Store','Year','Month'])
post = post[~post.index.isin(thin.index)].reset_index()
print(f"Giữ lại {len(days_per_sm)-len(thin)} tháng-CH sau khi loại các tháng quá thưa dữ liệu.")

grp = post.groupby(['Store','IsPromo2ActiveThisMonth'])['Sales'].mean().unstack().dropna()
grp.columns = ['inactive_month_sales','active_month_sales']
grp['pct_diff'] = (grp['active_month_sales'] - grp['inactive_month_sales']) / grp['inactive_month_sales'] * 100

fig, ax = plt.subplots(figsize=(7,4))
ax.hist(grp['pct_diff'], bins=40, color='#4472C4', edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.axvline(grp['pct_diff'].mean(), color='red', linestyle='--', label=f"TB = {grp['pct_diff'].mean():.2f}%")
ax.set_xlabel('% chênh lệch Sales (active - nghỉ), theo từng CH')
ax.set_ylabel('Số cửa hàng')
ax.set_title(f"n={len(grp)} CH, {(grp['pct_diff']>0).mean()*100:.1f}% CH có active > nghỉ")
ax.legend(); plt.tight_layout(); plt.show()
print(grp['pct_diff'].describe())


**Nhận xét:** Ngay cả trong cùng 1 CH, tháng active vẫn thấp hơn tháng nghỉ trung bình **~1.7%**, chỉ 25% CH có active > nghỉ. Nhìn thoáng qua, có vẻ củng cố "Promo2 gây hại" nhưng chưa loại trừ yếu tố mùa vụ.

### 3.5. Mùa vụ thị trường

In [ ]:
seasonal = df.groupby(df.Date.dt.month)['Sales'].mean()
overall_mean = df['Sales'].mean()
seasonal_idx = seasonal / overall_mean

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(seasonal_idx.index, seasonal_idx.values, marker='o', color='#4472C4')
ax.axhline(1.0, color='gray', linestyle=':')
ax.set_xticks(range(1,13)); ax.set_xlabel('Tháng'); ax.set_ylabel('Chỉ số mùa vụ (=1 là trung bình năm)')
ax.set_title('Chỉ số mùa vụ theo tháng - toàn bộ cửa hàng')
plt.tight_layout(); plt.show()
print(seasonal_idx.round(3))


**Nhận xét:** Doanh số có mùa vụ rõ rệt: tháng 12 cao vượt trội (chỉ số 1.24), tháng 1-2 và 8-10 thấp điểm (chỉ số ~0.94-0.96). Vì 3 chu kỳ PromoInterval chia 12 tháng thành 3 nhóm active khác nhau cho từng CH, phần "âm" ở mục 3.4 **rất có thể** chỉ là trùng hợp lịch, không phải do Promo2. Kiểm chứng bằng cách thuần mô tả (không fit model) ở bước tiếp theo.

### 3.6. Kiểm chứng thuần mô tả: nếu chỉ do lịch (mùa vụ), mức "âm" kỳ vọng là bao nhiêu?

In [ ]:
results = []
for pattern in store[store.Promo2==1]['PromoInterval'].dropna().unique():
    active_m = parse_interval(pattern)
    inactive_m = set(range(1,13)) - active_m
    active_idx = seasonal_idx[sorted(active_m)].mean()
    inactive_idx = seasonal_idx[sorted(inactive_m)].mean()
    n_stores_pattern = (store[store.Promo2==1]['PromoInterval']==pattern).sum()
    expected_gap_pct = (active_idx/inactive_idx - 1) * 100
    results.append((pattern, n_stores_pattern, active_idx, inactive_idx, expected_gap_pct))

res_df = pd.DataFrame(results, columns=['PromoInterval pattern','n CH','Seasonal idx (active)','Seasonal idx (nghỉ)','Expected gap % (chỉ do lịch)'])
print(res_df.round(3))

# weighted average theo đúng tỷ trọng n CH mỗi pattern -> con số dự đoán tổng thể
weighted_expected = (res_df['n CH'] * res_df['Expected gap % (chỉ do lịch)']).sum() / res_df['n CH'].sum()
actual_observed = grp['pct_diff'].mean()
print(f"\n>>> Gap dự đoán CHỈ từ trùng hợp lịch (mùa vụ), gộp theo tỷ trọng 3 pattern: {weighted_expected:+.2f}%")
print(f">>> Gap thực tế quan sát được ở mục 3.4 (within-store, active vs nghỉ)      : {actual_observed:+.2f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
names = ['Dự đoán chỉ từ\nmùa vụ (lịch)', 'Quan sát thực tế\n(within-store)']
vals = [weighted_expected, actual_observed]
bars = ax.bar(names, vals, color=['#70AD47','#C00000'])
ax.axhline(0, color='black', linewidth=0.8)
for b,v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v-0.3 if v<0 else v+0.1, f"{v:+.2f}%", ha='center')
ax.set_ylabel('% chênh lệch (active vs nghỉ)')
ax.set_title('Mùa vụ đơn thuần đã dự đoán gần đúng gap quan sát được')
plt.tight_layout(); plt.show()


**Đây là phát hiện quan trọng nhất của toàn bộ EDA:**

Chỉ cần dùng **chỉ số mùa vụ trung bình toàn thị trường** (mục 3.5) và biết CH nào rơi vào chu kỳ PromoInterval nào (mục 1.4), ta có thể **tính tay** mức chênh lệch "active vs nghỉ" mà thị trường lẽ ra sẽ có **dù Promo2 hoàn toàn không có tác dụng gì**. Chỉ vì sự trùng hợp giữa lịch chạy Promo2 và mùa cao/thấp điểm tự nhiên. Con số dự đoán thuần từ lịch này **gần khớp với gap thực tế quan sát được** ở mục 3.4.

→ Điều này cho thấy phần lớn "tác động tiêu cực" của Promo2 nhìn thấy ở mục 3.4 **có thể được giải thích chỉ bằng trùng hợp mùa vụ**, không cần viện đến giả thuyết "Promo2 có hại". Đây là bằng chứng mô tả (descriptive), đủ mạnh để đặt dấu hỏi lớn với kết luận vội vàng của HĐQT (việc kiểm định formal, tách bạch chính xác định lượng phần trăm do mùa vụ vs do Promo2, thuộc về giai đoạn Modeling ở M01, không nằm trong phạm vi EDA này).

---
## 4. Kết nối lại thành 1 câu chuyện — Vấn đề thật sự của doanh nghiệp là gì?

**Symptom bề mặt (điều HĐQT nhìn thấy):** 571 CH tham gia Promo2 có Sales TB thấp hơn 544 CH không tham gia ~10-11% → phản xạ tự nhiên là "cắt Promo2 để tiết kiệm chi phí".

**Nhưng khi đào sâu, lộ ra 2 root cause cộng dồn — không phải 1:**

**Root cause 1 — Cách ra quyết định hiện tại của doanh nghiệp đang dựa trên số liệu thô, dễ đọc sai:**
Toàn bộ hành trình điều tra ở mục 3 cho thấy: chỉ với 1 phép so sánh trung bình đơn giản (571 CH vs 544 CH), HĐQT đã suýt đi đến quyết định cắt cả 1 chương trình khuyến mãi — trong khi bằng chứng mô tả (3.2, 3.6) cho thấy phần lớn chênh lệch đó nhiều khả năng đến từ **composition CH khác nhau** + **trùng hợp mùa vụ**, không phải do Promo2 gây hại --> symptoms: **dữ liệu vận hành của Rossmann rất không đồng nhất** (StoreType/Assortment lệch mẫu nặng — mục 1.5; 259 CH không có trong tập cần dự báo — mục 1.6; 72 CH từng đóng cửa dài hạn làm méo mọi trung bình cuốn theo — mục 1.7; mùa vụ mạnh và không đều — mục 3.5), nhưng lại đang được **đọc bằng các phép so sánh trung bình đơn giản, không kiểm soát nhiễu**. Nếu cứ tiếp tục ra quyết định kiểu này, rủi ro không chỉ dừng ở Promo2 — bất kỳ quyết định chiến lược nào khác dựa trên "trung bình thô" đều có thể sai theo cách tương tự.

**Root cause 2 — Vận hành hiện tại không có 1 cách dự báo doanh số đáng tin cậy, nhất quán:**
Theo đúng bối cảnh gốc của bài toán (Rossmann/Kaggle): hiện tại, **từng store manager tự dự đoán doanh số 6 tuần tới cho cửa hàng của mình**, dựa trên kinh nghiệm/trực giác riêng — với hàng nghìn quản lý khác nhau, độ chính xác dự báo giữa các cửa hàng **rất không đồng đều**. Đây là root cause thứ 2, độc lập với Promo2: **doanh nghiệp không có 1 phương pháp dự báo tập trung, hệ thống hoá và nhất quán** — dẫn đến xếp ca nhân viên và nhập hàng không hiệu quả ở quy mô hơn 1.000 cửa hàng.

**2 root cause này cộng dồn lại thành 1 vấn đề duy nhất của doanh nghiệp:**
Rossmann hiện đang **ra quyết định (cả chiến lược lẫn vận hành) mà không có nền tảng phân tích dữ liệu đủ chặt chẽ** — quyết định chiến lược (Promo2) thì dựa vào số liệu thô dễ gây hiểu lầm; quyết định vận hành hàng ngày (dự báo) thì dựa vào cảm tính phân tán của hàng nghìn cá nhân. Cả 2 đều xuất phát từ cùng 1 gốc: **thiếu 1 quy trình phân tích dữ liệu tập trung, có kiểm soát nhiễu, áp dụng nhất quán trên toàn hệ thống 1.115 cửa hàng.**

**→ Từ đây, sự cần thiết của 2 mission trở nên hiển nhiên, không phải 2 việc rời rạc:**
- **Mission 1** giải quyết root cause 1: cần 1 phân tích **có kiểm soát nhiễu** (composition, mùa vụ, sample size) trước khi đưa ra khuyến nghị giữ/cắt Promo2 — thay vì tin ngay vào con số thô.
- **Mission 2** giải quyết root cause 2: cần 1 **mô hình dự báo tập trung, áp dụng đồng nhất** cho toàn bộ 856 CH có trong phạm vi test — thay thế cho việc hàng nghìn quản lý tự đoán rời rạc, để đảm bảo độ chính xác đồng đều và đáng tin cậy hơn khi lên kế hoạch nhập hàng/xếp ca.

Nói cách khác: **prediction (Mission 2) không chỉ là "cũng cần làm thêm" — nó là câu trả lời trực tiếp cho root cause 2, và cách làm nó (walk-forward, kiểm soát leakage, biết rõ phạm vi 856 CH) phải kế thừa đúng những gì phần EDA này đã phát hiện.** Tương tự, Mission 1 không thể làm bằng 1 phép so sánh trung bình nữa — phải kế thừa toàn bộ các lưu ý kiểm soát nhiễu ở mục 3.
